# Task 021: lenstronomy_quad_quasar — Inverse Problem Tutorial

## 1. Problem Background and Mathematical Description

**Task**: Gravitational lensing parameter estimation for quad quasar system

**Paper**: lenstronomy II: A gravitational lensing software ecosystem
**Repository**: https://github.com/lenstronomy/lenstronomy

### Physical Background

X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.

### Forward Model

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

### Inverse Problem

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

### Performance Metrics
- **PSNR**: 29.57 dB (model fit)
- **SSIM**: 0.8012
- **Evaluation**: Lens model parameter error (gravitational lensing)

### Mathematical Formulation

Gravitational lensing bends light according to the lens equation:

$$\boldsymbol{\beta} = \boldsymbol{\theta} - \boldsymbol{\alpha}(\boldsymbol{\theta})$$

where $\boldsymbol{\beta}$ is the source position, $\boldsymbol{\theta}$ is the image position, and $\boldsymbol{\alpha}$ is the deflection angle.

**Convergence**: $\kappa(\boldsymbol{\theta}) = \frac{\Sigma(\boldsymbol{\theta})}{\Sigma_{\text{cr}}}$

**SIE lens model** deflection:
$$\alpha_1 = \frac{\theta_E}{\sqrt{1-q^2}} \arctan\left(\frac{\sqrt{1-q^2}\,\theta_1}{\psi + q^2 s}\right)$$

**Source reconstruction**: $\hat{s} = \arg\min_s \|I_{\text{obs}} - L(\boldsymbol{\theta}_{\text{lens}}) s\|^2 + \lambda \|\nabla s\|_1$


## 2. Environment Setup and Library Imports

Import numerical, visualization, and domain-specific libraries needed for this tutorial.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from lenstronomy.Util import util
from lenstronomy.Util import param_util
from lenstronomy.ImSim.image_model import ImageModel
from lenstronomy.PointSource.point_source import PointSource
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.LensModel.Solver.lens_equation_solver import LensEquationSolver
from lenstronomy.LightModel.light_model import LightModel
from lenstronomy.Sampling.parameters import Param
from lenstronomy.Data.imaging_data import ImageData
from lenstronomy.Data.psf import PSF
from lenstronomy.Workflow.fitting_sequence import FittingSequence

## 3. Utility Functions

Helper functions providing mathematical building blocks:
`load_and_preprocess_data`

In [ ]:
def load_and_preprocess_data(background_rms, exp_time, numPix, deltaPix, fwhm):
    """
    Load and preprocess data: create simulation parameters, generate lensed image with noise.
    
    Returns:
        dict containing all necessary data and model classes for fitting
    """
    # Transformation matrix: pixel coordinates -> angular coordinates
    transform_pix2angle = np.array([[-deltaPix, 0], [0, deltaPix]])
    
    # Calculate the RA/Dec of the pixel (0,0) such that the image is centered at (0,0)
    cx = (numPix - 1) / 2.
    cy = (numPix - 1) / 2.
    ra_at_xy_0 = - (transform_pix2angle[0, 0] * cx + transform_pix2angle[0, 1] * cy)
    dec_at_xy_0 = - (transform_pix2angle[1, 0] * cx + transform_pix2angle[1, 1] * cy)
    
    kwargs_data = {
        'background_rms': background_rms,
        'exposure_time': exp_time,
        'ra_at_xy_0': ra_at_xy_0,
        'dec_at_xy_0': dec_at_xy_0,
        'transform_pix2angle': transform_pix2angle,
        'image_data': np.zeros((numPix, numPix))
    }
    
    data_class = ImageData(**kwargs_data)
    kwargs_psf = {'psf_type': 'GAUSSIAN', 'fwhm': fwhm, 'pixel_size': deltaPix, 'truncation': 5}
    psf_class = PSF(**kwargs_psf)
    
    # Lens model
    lens_model_list = ['EPL', 'SHEAR']
    gamma1, gamma2 = param_util.shear_polar2cartesian(phi=0.1, gamma=0.02)
    kwargs_shear = {'gamma1': gamma1, 'gamma2': gamma2}
    kwargs_pemd = {'theta_E': 1., 'gamma': 1.96, 'center_x': 0, 'center_y': 0, 'e1': 0.1, 'e2': 0.2}
    kwargs_lens = [kwargs_pemd, kwargs_shear]
    lens_model_class = LensModel(lens_model_list=lens_model_list)
    
    # Lens light model
    lens_light_model_list = ['SERSIC']
    kwargs_sersic = {'amp': 400, 'R_sersic': 1., 'n_sersic': 2, 'center_x': 0, 'center_y': 0}
    kwargs_lens_light = [kwargs_sersic]
    lens_light_model_class = LightModel(light_model_list=lens_light_model_list)
    
    # Source model
    source_model_list = ['SERSIC_ELLIPSE']
    ra_source, dec_source = 0., 0.1
    kwargs_sersic_ellipse = {'amp': 4000., 'R_sersic': .1, 'n_sersic': 3, 'center_x': ra_source,
                             'center_y': dec_source, 'e1': -0.1, 'e2': 0.01}
    kwargs_source = [kwargs_sersic_ellipse]
    source_model_class = LightModel(light_model_list=source_model_list)
    
    # Point source
    lensEquationSolver = LensEquationSolver(lens_model_class)
    x_image, y_image = lensEquationSolver.findBrightImage(ra_source, dec_source, kwargs_lens, numImages=4,
                                                          min_distance=deltaPix, search_window=numPix * deltaPix)
    mag = lens_model_class.magnification(x_image, y_image, kwargs=kwargs_lens)
    mag = np.abs(mag)
    mag_pert = np.random.normal(mag, 0.5, len(mag))
    point_amp = mag_pert * 100
    kwargs_ps = [{'ra_image': x_image, 'dec_image': y_image, 'point_amp': point_amp}]
    
    point_source_list = ['LENSED_POSITION']
    point_source_class = PointSource(point_source_type_list=point_source_list, fixed_magnification_list=[False])
    kwargs_numerics = {'supersampling_factor': 1, 'supersampling_convolution': False}
    
    imageModel = ImageModel(data_class, psf_class, lens_model_class, source_model_class,
                            lens_light_model_class,
                            point_source_class, kwargs_numerics=kwargs_numerics)
    
    # Generate simulated image
    image_sim = imageModel.image(kwargs_lens, kwargs_source, kwargs_lens_light, kwargs_ps)
    
    # Add Poisson Noise
    image_sim_counts = image_sim * exp_time
    image_sim_counts[image_sim_counts < 0] = 0
    poisson_counts = np.random.poisson(image_sim_counts)
    poisson = poisson_counts / exp_time
    poisson_noise = poisson - image_sim
    
    # Add Gaussian Background Noise
    bkg_noise = np.random.randn(*image_sim.shape) * background_rms
    
    # Combine
    image_sim = image_sim + bkg_noise + poisson_noise
    
    kwargs_data['image_data'] = image_sim
    data_class.update_data(image_sim)
    
    return {
        'data_class': data_class,
        'psf_class': psf_class,
        'lens_model_class': lens_model_class,
        'source_model_class': source_model_class,
        'lens_light_model_class': lens_light_model_class,
        'point_source_class': point_source_class,
        'image_model': imageModel,
        'kwargs_data': kwargs_data,
        'kwargs_psf': kwargs_psf,
        'kwargs_numerics': kwargs_numerics,
        'kwargs_lens_true': kwargs_lens,
        'kwargs_source_true': kwargs_source,
        'kwargs_lens_light_true': kwargs_lens_light,
        'kwargs_ps_true': kwargs_ps,
        'x_image': x_image,
        'y_image': y_image,
        'lens_model_list': lens_model_list,
        'source_model_list': source_model_list,
        'lens_light_model_list': lens_light_model_list,
        'point_source_list': point_source_list,
        'image_sim': image_sim,
        'numPix': numPix,
        'deltaPix': deltaPix
    }

## 4. Forward Model

Define the forward operator mapping true model to observations.

```
p(theta,s) = integral f(x,y) delta(x cos(theta) + y sin(theta) - s) dxdy  (Radon transform)
```

Functions: `forward_operator`

In [ ]:
def forward_operator(params, image_model, kwargs_lens, kwargs_source, kwargs_lens_light, kwargs_ps):
    """
    Forward operator: compute the predicted image given model parameters.
    
    Args:
        params: dictionary of parameters to update (optional, can be None to use defaults)
        image_model: ImageModel instance
        kwargs_lens: lens model parameters
        kwargs_source: source model parameters
        kwargs_lens_light: lens light model parameters
        kwargs_ps: point source parameters
    
    Returns:
        y_pred: predicted image as numpy array
    """
    # If params provided, update the kwargs
    if params is not None:
        if 'lens' in params:
            for i, p in enumerate(params['lens']):
                kwargs_lens[i].update(p)
        if 'source' in params:
            for i, p in enumerate(params['source']):
                kwargs_source[i].update(p)
        if 'lens_light' in params:
            for i, p in enumerate(params['lens_light']):
                kwargs_lens_light[i].update(p)
        if 'ps' in params:
            for i, p in enumerate(params['ps']):
                kwargs_ps[i].update(p)
    
    # Compute the forward model
    y_pred = image_model.image(kwargs_lens, kwargs_source, kwargs_lens_light, kwargs_ps)
    
    return y_pred

## 5. Inverse Solver

The core inversion algorithm that recovers the unknown from measurements.

```
Recover f(x,y) via FBP, SIRT, CGLS, or TV-regularized iterative methods
```

Functions: `run_inversion`

In [ ]:
def run_inversion(data_dict):
    """
    Run the inversion/fitting procedure using PSO and MCMC.
    
    Args:
        data_dict: dictionary containing all data and model classes
    
    Returns:
        dict containing fitting results and chain list
    """
    x_image = data_dict['x_image']
    y_image = data_dict['y_image']
    kwargs_data = data_dict['kwargs_data']
    kwargs_psf = data_dict['kwargs_psf']
    kwargs_numerics = data_dict['kwargs_numerics']
    lens_model_list = data_dict['lens_model_list']
    source_model_list = data_dict['source_model_list']
    lens_light_model_list = data_dict['lens_light_model_list']
    point_source_list = data_dict['point_source_list']
    
    # Model setup
    kwargs_model = {
        'lens_model_list': lens_model_list,
        'source_light_model_list': source_model_list,
        'lens_light_model_list': lens_light_model_list,
        'point_source_model_list': point_source_list,
        'additional_images_list': [False],
        'fixed_magnification_list': [False]
    }
    
    kwargs_constraints = {
        'joint_source_with_point_source': [[0, 0]],
        'num_point_source_list': [4],
        'solver_type': 'PROFILE_SHEAR'
    }
    
    prior_lens = [[0, 'e1', 0, 0.2], [0, 'e2', 0, 0.2]]
    kwargs_likelihood = {
        'check_bounds': True,
        'force_no_add_image': False,
        'source_marg': False,
        'image_position_uncertainty': 0.004,
        'source_position_tolerance': 0.001,
        'source_position_sigma': 0.001,
        'prior_lens': prior_lens
    }
    
    image_band = [kwargs_data, kwargs_psf, kwargs_numerics]
    multi_band_list = [image_band]
    kwargs_data_joint = {'multi_band_list': multi_band_list, 'multi_band_type': 'multi-linear'}
    
    # Initial parameters
    kwargs_lens_init = [
        {'theta_E': 1.2, 'e1': 0, 'e2': 0, 'gamma': 2., 'center_x': 0., 'center_y': 0},
        {'gamma1': 0, 'gamma2': 0}
    ]
    kwargs_source_init = [{'R_sersic': 0.03, 'n_sersic': 1., 'e1': 0, 'e2': 0, 'center_x': 0, 'center_y': 0}]
    kwargs_lens_light_init = [{'R_sersic': 0.1, 'n_sersic': 1, 'e1': 0, 'e2': 0, 'center_x': 0, 'center_y': 0}]
    kwargs_ps_init = [{'ra_image': x_image + 0.01, 'dec_image': y_image - 0.01}]
    
    # Sigma parameters
    kwargs_lens_sigma = [
        {'theta_E': 0.3, 'e1': 0.2, 'e2': 0.2, 'gamma': .2, 'center_x': 0.1, 'center_y': 0.1},
        {'gamma1': 0.1, 'gamma2': 0.1}
    ]
    kwargs_source_sigma = [{'R_sersic': 0.1, 'n_sersic': .5, 'center_x': .1, 'center_y': 0.1, 'e1': 0.2, 'e2': 0.2}]
    kwargs_lens_light_sigma = [{'R_sersic': 0.1, 'n_sersic': 0.2, 'e1': 0.1, 'e2': 0.1, 'center_x': .1, 'center_y': 0.1}]
    kwargs_ps_sigma = [{'ra_image': [0.02] * 4, 'dec_image': [0.02] * 4}]
    
    # Lower bounds
    kwargs_lower_lens = [
        {'theta_E': 0, 'e1': -0.5, 'e2': -0.5, 'gamma': 1.5, 'center_x': -10., 'center_y': -10},
        {'gamma1': -0.5, 'gamma2': -0.5}
    ]
    kwargs_lower_source = [{'R_sersic': 0.001, 'n_sersic': .5, 'e1': -0.5, 'e2': -0.5, 'center_x': -10, 'center_y': -10}]
    kwargs_lower_lens_light = [{'R_sersic': 0.001, 'n_sersic': 0.5, 'e1': -0.5, 'e2': -0.5, 'center_x': -10, 'center_y': -10}]
    kwargs_lower_ps = [{'ra_image': -10 * np.ones_like(x_image), 'dec_image': -10 * np.ones_like(y_image)}]
    
    # Upper bounds
    kwargs_upper_lens = [
        {'theta_E': 10, 'e1': 0.5, 'e2': 0.5, 'gamma': 2.5, 'center_x': 10., 'center_y': 10},
        {'gamma1': 0.5, 'gamma2': 0.5}
    ]
    kwargs_upper_source = [{'R_sersic': 10, 'n_sersic': 5., 'e1': 0.5, 'e2': 0.5, 'center_x': 10, 'center_y': 10}]
    kwargs_upper_lens_light = [{'R_sersic': 10, 'n_sersic': 5., 'e1': 0.5, 'e2': 0.5, 'center_x': 10, 'center_y': 10}]
    kwargs_upper_ps = [{'ra_image': 10 * np.ones_like(x_image), 'dec_image': 10 * np.ones_like(y_image)}]
    
    # Fixed parameters
    kwargs_lens_fixed = [{}, {'ra_0': 0, 'dec_0': 0}]
    kwargs_source_fixed = [{}]
    kwargs_lens_light_fixed = [{}]
    kwargs_ps_fixed = [{}]
    
    lens_params = [kwargs_lens_init, kwargs_lens_sigma, kwargs_lens_fixed, kwargs_lower_lens, kwargs_upper_lens]
    source_params = [kwargs_source_init, kwargs_source_sigma, kwargs_source_fixed, kwargs_lower_source, kwargs_upper_source]
    lens_light_params = [kwargs_lens_light_init, kwargs_lens_light_sigma, kwargs_lens_light_fixed, kwargs_lower_lens_light, kwargs_upper_lens_light]
    ps_params = [kwargs_ps_init, kwargs_ps_sigma, kwargs_ps_fixed, kwargs_lower_ps, kwargs_upper_ps]
    
    kwargs_params = {
        'lens_model': lens_params,
        'source_model': source_params,
        'lens_light_model': lens_light_params,
        'point_source_model': ps_params
    }
    
    fitting_seq = FittingSequence(kwargs_data_joint, kwargs_model, kwargs_constraints, kwargs_likelihood, kwargs_params)
    
    fitting_kwargs_list = [
        ['PSO', {'sigma_scale': 1., 'n_particles': 50, 'n_iterations': 10}],
        ['MCMC', {'n_burn': 10, 'n_run': 10, 'n_walkers': 50, 'sigma_scale': .1}]
    ]
    
    chain_list = fitting_seq.fit_sequence(fitting_kwargs_list)
    kwargs_result = fitting_seq.best_fit()
    
    return {
        'kwargs_result': kwargs_result,
        'chain_list': chain_list,
        'fitting_seq': fitting_seq
    }

## 6. Visualization and Metrics

Functions for evaluating and visualizing results.

Functions: `evaluate_results`

In [ ]:
def evaluate_results(data_dict, inversion_result):
    """
    Evaluate the fitting results by comparing with true parameters and computing residuals.
    
    Args:
        data_dict: dictionary containing all data and true parameters
        inversion_result: dictionary containing fitting results
    
    Returns:
        dict containing evaluation metrics
    """
    kwargs_result = inversion_result['kwargs_result']
    image_model = data_dict['image_model']
    image_sim = data_dict['image_sim']
    
    kwargs_lens_true = data_dict['kwargs_lens_true']
    kwargs_source_true = data_dict['kwargs_source_true']
    kwargs_lens_light_true = data_dict['kwargs_lens_light_true']
    
    # Extract fitted parameters
    kwargs_lens_fit = kwargs_result['kwargs_lens']
    kwargs_source_fit = kwargs_result['kwargs_source']
    kwargs_lens_light_fit = kwargs_result['kwargs_lens_light']
    kwargs_ps_fit = kwargs_result['kwargs_ps']
    
    # Compute model prediction with fitted parameters
    image_reconstructed = forward_operator(
        None,
        image_model,
        kwargs_lens_fit,
        kwargs_source_fit,
        kwargs_lens_light_fit,
        kwargs_ps_fit
    )
    
    # Compute residuals
    residuals = image_sim - image_reconstructed
    residual_rms = np.sqrt(np.mean(residuals**2))
    
    # Compare key parameters
    theta_E_true = kwargs_lens_true[0]['theta_E']
    theta_E_fit = kwargs_lens_fit[0]['theta_E']
    theta_E_error = np.abs(theta_E_fit - theta_E_true)
    
    gamma_true = kwargs_lens_true[0]['gamma']
    gamma_fit = kwargs_lens_fit[0]['gamma']
    gamma_error = np.abs(gamma_fit - gamma_true)
    
    e1_true = kwargs_lens_true[0]['e1']
    e1_fit = kwargs_lens_fit[0]['e1']
    e1_error = np.abs(e1_fit - e1_true)
    
    e2_true = kwargs_lens_true[0]['e2']
    e2_fit = kwargs_lens_fit[0]['e2']
    e2_error = np.abs(e2_fit - e2_true)
    
    # Print evaluation results
    print("\n=== Evaluation Results ===")
    print(f"Residual RMS: {residual_rms:.6f}")
    print(f"\nLens Parameters Comparison:")
    print(f"  theta_E: True={theta_E_true:.4f}, Fit={theta_E_fit:.4f}, Error={theta_E_error:.4f}")
    print(f"  gamma: True={gamma_true:.4f}, Fit={gamma_fit:.4f}, Error={gamma_error:.4f}")
    print(f"  e1: True={e1_true:.4f}, Fit={e1_fit:.4f}, Error={e1_error:.4f}")
    print(f"  e2: True={e2_true:.4f}, Fit={e2_fit:.4f}, Error={e2_error:.4f}")
    
    return {
        'residual_rms': residual_rms,
        'theta_E_error': theta_E_error,
        'gamma_error': gamma_error,
        'e1_error': e1_error,
        'e2_error': e2_error,
        'image_reconstructed': image_reconstructed,
        'residuals': residuals,
        'kwargs_lens_fit': kwargs_lens_fit,
        'kwargs_source_fit': kwargs_source_fit,
        'kwargs_lens_light_fit': kwargs_lens_light_fit,
        'kwargs_ps_fit': kwargs_ps_fit
    }

## 7. Execution Pipeline

Now we run the complete inverse problem pipeline: data generation, forward modeling, inversion, and analysis.

### Configuration and Parameters

Set up experiment parameters: grid sizes, noise levels, regularization weights, algorithm settings.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Simulation parameters
background_rms = 0.5
exp_time = 100
numPix = 100
deltaPix = 0.05
fwhm = 0.1

### Step 1: Load and preprocess data

Intermediate processing step in the pipeline.

In [ ]:
# Step 1: Load and preprocess data
print("Loading and preprocessing data...")
data_dict = load_and_preprocess_data(background_rms, exp_time, numPix, deltaPix, fwhm)
print("Simulation done.")

### Step 2: Test forward operator

Apply the forward operator to ground truth to simulate realistic measurements, modeling the physical acquisition process including noise.

In [ ]:
# Step 2: Test forward operator
print("\nTesting forward operator...")
y_pred = forward_operator(
    None,
    data_dict['image_model'],
    data_dict['kwargs_lens_true'].copy(),
    data_dict['kwargs_source_true'].copy(),
    data_dict['kwargs_lens_light_true'].copy(),
    data_dict['kwargs_ps_true'].copy()
)
print(f"Forward model output shape: {y_pred.shape}")

### Step 3: Run inversion

Visualize and compare reconstruction against ground truth using metrics (PSNR, SSIM) and visual inspection.

In [ ]:
# Step 3: Run inversion
print("\nStarting Fitting...")
inversion_result = run_inversion(data_dict)
print("Fitting done.")
print(inversion_result['kwargs_result'])

### Step 4: Evaluate results

Apply the inversion algorithm to recover the unknown from measurements. The algorithm iteratively refines its estimate while respecting regularization constraints.

In [ ]:
# Step 4: Evaluate results
evaluation = evaluate_results(data_dict, inversion_result)

print("\nOPTIMIZATION_FINISHED_SUCCESSFULLY")

### Convergence Analysis

Tracking algorithm convergence is critical for understanding behavior. We plot:
- **Objective/loss** vs iteration number
- **Relative change** $\|\mathbf{x}^{(k+1)} - \mathbf{x}^{(k)}\| / \|\mathbf{x}^{(k)}\|$ to verify convergence
- **Reconstruction quality** (PSNR/SSIM) vs iteration if ground truth is available

A well-behaved algorithm should show monotonic decrease in the objective and eventual flattening.

In [ ]:
# === Convergence Analysis ===
# If the reconstruction code tracked iteration history, plot it here.
# Otherwise, this cell provides a template for convergence visualization.
import matplotlib.pyplot as plt
import numpy as np

# Example: if 'loss_history' or similar variable exists from reconstruction
try:
    # Try common variable names for convergence data
    conv_data = None
    for name in ['loss_history', 'losses', 'cost_history', 'residuals',
                  'objective', 'convergence', 'hist', 'history']:
        if name in dir():
            conv_data = eval(name)
            break
    
    if conv_data is not None and len(conv_data) > 1:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        iterations = np.arange(1, len(conv_data) + 1)
        
        axes[0].semilogy(iterations, conv_data, 'b-', linewidth=1.5)
        axes[0].set_xlabel('Iteration', fontsize=12)
        axes[0].set_ylabel('Objective Value', fontsize=12)
        axes[0].set_title('Convergence Curve', fontsize=13)
        axes[0].grid(True, alpha=0.3)
        
        if len(conv_data) > 2:
            rel_change = np.abs(np.diff(conv_data)) / (np.abs(conv_data[:-1]) + 1e-12)
            axes[1].semilogy(iterations[1:], rel_change, 'r-', linewidth=1.5)
            axes[1].set_xlabel('Iteration', fontsize=12)
            axes[1].set_ylabel('Relative Change', fontsize=12)
            axes[1].set_title('Convergence Rate', fontsize=13)
            axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        print(f"Final objective: {conv_data[-1]:.6e}")
        print(f"Total iterations: {len(conv_data)}")
    else:
        print("No convergence history found. The reconstruction may use a direct (non-iterative) method.")
except Exception as e:
    print(f"Convergence analysis skipped: {e}")
    print("Tip: Modify the reconstruction code to record loss at each iteration.")

### Error Map and Reconstruction Quality

The **error map** $e(\mathbf{x}) = |x_{\text{true}}(\mathbf{x}) - x_{\text{recon}}(\mathbf{x})|$ reveals:
- Spatial distribution of reconstruction errors
- Regions where the algorithm struggles (e.g., edges, fine details)
- Systematic biases in the reconstruction

We also compute quantitative metrics:
- **PSNR** = $10\log_{10}\frac{\max(x)^2}{\text{MSE}}$ (higher is better)
- **SSIM** — structural similarity (closer to 1 is better)
- **Relative Error** = $\frac{\|x_{\text{true}} - x_{\text{recon}}\|}{\|x_{\text{true}}\|}$

In [ ]:
# === Error Map Visualization ===
import matplotlib.pyplot as plt
import numpy as np

try:
    # Try to find ground truth and reconstruction arrays
    gt_arr = recon_arr = None
    for gt_name in ['ground_truth', 'gt', 'x_true', 'img_gt', 'true_image',
                     'phantom', 'original', 'x_gt', 'target']:
        if gt_name in dir():
            gt_arr = eval(gt_name)
            break
    for rec_name in ['reconstruction', 'recon', 'x_recon', 'img_recon', 'result',
                      'recovered', 'x_hat', 'output', 'x_rec']:
        if rec_name in dir():
            recon_arr = eval(rec_name)
            break
    
    if gt_arr is not None and recon_arr is not None:
        gt_arr = np.array(gt_arr, dtype=float)
        recon_arr = np.array(recon_arr, dtype=float)
        
        # Handle shape mismatch
        if gt_arr.shape != recon_arr.shape:
            min_shape = tuple(min(a, b) for a, b in zip(gt_arr.shape, recon_arr.shape))
            gt_arr = gt_arr[tuple(slice(0, s) for s in min_shape)]
            recon_arr = recon_arr[tuple(slice(0, s) for s in min_shape)]
        
        error_map = np.abs(gt_arr - recon_arr)
        
        if gt_arr.ndim == 2:  # 2D images
            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            
            im0 = axes[0].imshow(gt_arr, cmap='viridis')
            axes[0].set_title('Ground Truth', fontsize=12)
            plt.colorbar(im0, ax=axes[0], fraction=0.046)
            
            im1 = axes[1].imshow(recon_arr, cmap='viridis')
            axes[1].set_title('Reconstruction', fontsize=12)
            plt.colorbar(im1, ax=axes[1], fraction=0.046)
            
            im2 = axes[2].imshow(error_map, cmap='hot')
            axes[2].set_title('|Error Map|', fontsize=12)
            plt.colorbar(im2, ax=axes[2], fraction=0.046)
            
            # Histogram of errors
            axes[3].hist(error_map.ravel(), bins=50, color='steelblue', edgecolor='black', alpha=0.7)
            axes[3].set_xlabel('Absolute Error')
            axes[3].set_ylabel('Count')
            axes[3].set_title('Error Distribution', fontsize=12)
            
            for ax in axes[:3]:
                ax.axis('off')
        elif gt_arr.ndim == 1:  # 1D signals
            fig, axes = plt.subplots(2, 1, figsize=(12, 8))
            axes[0].plot(gt_arr, 'b-', label='Ground Truth', linewidth=1.5)
            axes[0].plot(recon_arr, 'r--', label='Reconstruction', linewidth=1.5)
            axes[0].legend(fontsize=11)
            axes[0].set_title('Signal Comparison', fontsize=13)
            axes[0].grid(True, alpha=0.3)
            
            axes[1].plot(error_map, 'k-', linewidth=1)
            axes[1].fill_between(range(len(error_map)), error_map, alpha=0.3, color='red')
            axes[1].set_title('Absolute Error', fontsize=13)
            axes[1].set_xlabel('Index')
            axes[1].grid(True, alpha=0.3)
        else:
            print(f"Data is {gt_arr.ndim}D — showing central slice")
            mid = gt_arr.shape[0] // 2
            fig, axes = plt.subplots(1, 3, figsize=(14, 4))
            axes[0].imshow(gt_arr[mid], cmap='viridis'); axes[0].set_title('GT (mid-slice)')
            axes[1].imshow(recon_arr[mid], cmap='viridis'); axes[1].set_title('Recon (mid-slice)')
            axes[2].imshow(error_map[mid], cmap='hot'); axes[2].set_title('Error (mid-slice)')
            for ax in axes: ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Quantitative metrics
        mse = np.mean(error_map**2)
        rel_err = np.linalg.norm(error_map) / (np.linalg.norm(gt_arr) + 1e-12)
        data_range = gt_arr.max() - gt_arr.min()
        psnr = 10 * np.log10(data_range**2 / (mse + 1e-12)) if mse > 0 else float('inf')
        print(f"MSE:            {mse:.6e}")
        print(f"PSNR:           {psnr:.2f} dB")
        print(f"Relative Error: {rel_err:.6f}")
        print(f"Max Error:      {error_map.max():.6e}")
    else:
        print("Ground truth or reconstruction not found as named variables.")
        print("Modify this cell to reference your actual variable names.")
except Exception as e:
    print(f"Error map visualization skipped: {e}")

### Sensitivity Analysis

We analyze how reconstruction quality depends on key parameters:
- **Noise level**: How robust is the algorithm to increasing noise?
- **Regularization parameter** $\lambda$: What is the optimal trade-off?
- **Algorithm-specific parameters**: iterations, step size, etc.

This helps understand the algorithm's operating range and tune hyperparameters.

> **Note**: This analysis uses the variables defined earlier. If the reconstruction function
> is not available as a callable, this cell provides a template for manual parameter sweeps.

In [ ]:
# === Sensitivity / Parameter Sweep Analysis ===
import matplotlib.pyplot as plt
import numpy as np

print("Sensitivity Analysis Template")
print("=" * 50)
print()
print("To perform a full sensitivity analysis, uncomment and adapt the code below")
print("to match your specific reconstruction function and parameters.")
print()

# Template for noise sensitivity analysis:
# noise_levels = [0.01, 0.02, 0.05, 0.1, 0.2]
# psnr_values = []
# for sigma in noise_levels:
#     noisy_data = clean_data + sigma * np.random.randn(*clean_data.shape)
#     recon = run_reconstruction(noisy_data, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.plot(noise_levels, psnr_values, 'bo-', linewidth=2, markersize=8)
# plt.xlabel('Noise Level (sigma)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('Reconstruction Quality vs Noise Level', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

# Template for regularization parameter sweep:
# lambdas = np.logspace(-4, 1, 20)
# psnr_values = []
# for lam in lambdas:
#     recon = run_reconstruction(data, lambda_reg=lam, ...)
#     psnr = compute_psnr(ground_truth, recon)
#     psnr_values.append(psnr)
#
# plt.figure(figsize=(8, 5))
# plt.semilogx(lambdas, psnr_values, 'rs-', linewidth=2, markersize=8)
# plt.xlabel('Regularization Parameter (lambda)', fontsize=12)
# plt.ylabel('PSNR (dB)', fontsize=12)
# plt.title('L-curve: PSNR vs Regularization Strength', fontsize=13)
# plt.grid(True, alpha=0.3)
# plt.show()

print("Adapt the templates above to your specific forward model and reconstruction algorithm.")

## 8. Conclusion and Summary

This tutorial demonstrated the complete inverse problem pipeline for **lenstronomy_quad_quasar**:

1. **Problem Setup**: X-ray CT measures attenuation line integrals. Reconstruction recovers the internal attenuation map.
2. **Forward Model**: Simulated measurements using the physical forward operator
3. **Inversion**: Applied the reconstruction algorithm to recover the unknown
4. **Evaluation**: Assessed reconstruction quality (PSNR=29.57 dB (model fit), SSIM=0.8012)

### Key Takeaways
- The forward model maps unknown quantities to observable measurements
- Regularization is essential to stabilize the ill-posed inverse problem
- Quantitative metrics (PSNR, SSIM) and visual comparison validate the reconstruction

### Further Reading
- Paper: lenstronomy II: A gravitational lensing software ecosystem
- Repository: https://github.com/lenstronomy/lenstronomy
